# Imports

In [ ]:
import glob
import os

import kaleido
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

kaleido.get_chrome_sync()

# Load data

In [ ]:
def load_df(path):
    df = pd.read_csv(path)
    split = path.split("_")
    if len(split) == 6:
        unsupervised = True
    elif len(split) == 5:
        unsupervised = False
    else:
        raise ValueError(f"Invalid path {path}")
        
    df["window_size"] = split[-3]
    df["unsupervised"] = unsupervised

    return df

## Get TS_UCF results

In [ ]:
notebooks_dir = os.path.abspath("")
res_dir = os.path.join(os.path.dirname(notebooks_dir), "results")
figures_dir = os.path.join(os.path.dirname(notebooks_dir), "figures")
all_paths = glob.glob(f"{res_dir}/TS_UCF_*_evaluation_results.csv")
print(all_paths)
df_concat = pd.concat([load_df(path) for path in all_paths], ignore_index=True)
df_concat["algo"] = "TS_UCF_" + df_concat["unsupervised"].map({False : "", True: "U_"}).astype(str) + df_concat["window_size"].astype(str) 

In [ ]:
# Compute best_VUS_PR version
idx = df_concat.groupby("data_file")["VUS-PR"].idxmax()
best_rows = df_concat.loc[idx].copy()
best_rows["algo"] = "TS_UCF_best_VUS_PR"

# Compute best_AUC_ROC version
idx_roc = df_concat.groupby("data_file")["AUC-ROC"].idxmax()
best_rows_roc = df_concat.loc[idx_roc].copy()
best_rows_roc["algo"] = "TS_UCF_best_AUC_ROC"

df_concat = pd.concat([df_concat, best_rows, best_rows_roc], ignore_index=True)

In [ ]:
ordered_algo = df_concat.groupby("algo")["VUS-PR"].mean().sort_values(ascending=True).index.tolist()
df_concat["algo"] = pd.Categorical(df_concat["algo"], categories=ordered_algo, ordered=True)
df_concat = df_concat.sort_values("algo")

## Get mTSBench results

In [ ]:
df_res_mtsbench = pd.read_csv(os.path.join(res_dir, "auc_roc_pivot_mtsbench.csv")).set_index("data_folder")
df_res_mtsbench.columns.name = "algo"

In [ ]:
vus_pr_mtsbench_data = [
    # (Name,        min, q1, med,   q3,   max)
    ("PCA",         0, 0.10, 0.28, 0.54, 1.00),
    ("OmniAnomaly", 0, 0.09, 0.28, 0.53, 1.00),
    ("OFA",         0, 0.09, 0.29, 0.52, 1.00),
    ("CNN",         0, 0.08, 0.21, 0.48, 1.00),
    ("IForest",     0, 0.07, 0.19, 0.44, 1.00),
    ("ALLM4TS",     0, 0.05, 0.18, 0.47, 1.00),
    ("LSTMAD",      0, 0.07, 0.19, 0.43, 0.96),
    ("KMeansAD",    0, 0.07, 0.18, 0.41, 0.92),
    ("TranAD",      0, 0.05, 0.16, 0.40, 0.92),
    ("USAD",        0, 0.05, 0.17, 0.38, 0.88),
    ("MCD",         0, 0.04, 0.14, 0.33, 0.77),
    ("AutoEncoder", 0, 0.05, 0.14, 0.33, 0.77),
    ("TimesNet",    0, 0.05, 0.17, 0.37, 0.82),
    ("Donut",       0, 0.05, 0.14, 0.37, 0.87),
    ("FITS",        0, 0.05, 0.16, 0.34, 0.73),
    ("OCSVM",       0, 0.04, 0.11, 0.30, 0.70),
    ("CBLOF",       0, 0.05, 0.13, 0.33, 0.74),
    ("EIF",         0, 0.05, 0.12, 0.30, 0.68),
    ("RobustPCA",   0, 0.03, 0.09, 0.26, 0.62),
    ("COPOD",       0, 0.04, 0.10, 0.29, 0.65),
    ("HBOS",        0, 0.04, 0.10, 0.27, 0.62),
    ("KNN",         0, 0.04, 0.11, 0.30, 0.67),
    ("LOF",         0, 0.04, 0.11, 0.25, 0.58),
    ("Transformer", 0, 0.03, 0.08, 0.21, 0.46),
]


# Plot functions

In [ ]:
WIDTH = 1500
HEIGHT = 990

FONT_STYLE = dict(
    family="Times New Roman",
    size=14,
    color="black"
)

In [ ]:
def plot_scaled_matrix(pivot: pd.DataFrame, cmap="inferno_r", log_scale=False, name=None):
    row_mins = pivot.min(axis=1).to_numpy()
    row_maxs = pivot.max(axis=1).to_numpy()
    row_ranges = np.where(row_maxs - row_mins == 0, 1, row_maxs - row_mins)
    scaled_pivot = (pivot - row_mins.reshape(-1, 1)) / row_ranges.reshape(-1, 1)

    column_means = pivot.mean(axis=0)

    sorted_columns = column_means.sort_values().index
    pivot_sorted = pivot[sorted_columns]
    scaled_pivot_sorted = scaled_pivot[sorted_columns]


    tick_text = []
    color_bar = []
    for text in sorted_columns:
        if "TS_UCF" in text:
            tick_text.append(f'<span style="color: red;">{text}</span>')
            color_bar.append("red")
        else:
            tick_text.append(f'<span style="color: black;">{text}</span>')
            color_bar.append("blue")

    fig = make_subplots(rows=2, cols=1, row_heights=[100, 800], vertical_spacing=0.03)

    if not log_scale:
        fig.add_trace(
            go.Bar(x=column_means[sorted_columns].index, y=column_means[sorted_columns].values, text=[f'{value:.2f}' for value in column_means[sorted_columns].values], marker_color=color_bar),
            row=1, col=1
        )

    else:
        columns_med = pivot.median(axis=0)
        fig.add_trace(
            go.Bar(x=columns_med[sorted_columns].index, y=columns_med[sorted_columns].values, text=[f'{value:.2f}' for value in columns_med[sorted_columns].values], marker_color=color_bar),
            row=1, col=1
        )

    if log_scale:
        fig.update_yaxes(type="log", row=1, col=1)

    fig.add_trace(
        go.Heatmap(
            colorbar={"title": "AUC-ROC" if not log_scale else "Runtime (s)"},
            z=scaled_pivot_sorted,
            text=pivot_sorted.values,
            texttemplate="%{text:.2f}",
            colorscale=cmap,
            hoverinfo="text",
            x=sorted_columns,
            y=pivot_sorted.index
        ),
        row=2, col=1
    )


    fig.update_layout(template="plotly_white", width=WIDTH, height=HEIGHT-100, font=FONT_STYLE, margin=dict(l=55, r=15, t=15, b=45), showlegend=False)
    
    fig.update_xaxes(showticklabels=False, row=1, col=1)
    fig.update_yaxes(title_text="Average", row=1, col=1, showline=True, linecolor="black", ticks="inside", mirror=True, gridcolor="#E5E5E5", gridwidth=0.5)

    fig.update_xaxes(side="bottom", tickmode='array', tickvals=list(range(len(sorted_columns))), ticktext=tick_text, row=2, col=1, showline=True, linecolor='black', tickangle=-45, ticks='outside', showgrid=False)
    fig.update_yaxes(title_text="Dataset", tickmode='array', tickvals=list(range(len(pivot_sorted.index))), ticktext=pivot_sorted.index, row=2, col=1, showline=True, linecolor='black', ticks='outside', showgrid=False)

    if name is not None:
        fig.write_image(f"{name}.pdf", format="pdf")
        
    fig.show()

In [ ]:
def show_box(data, name=None):
    names   = [d[0] for d in data]
    lower   = [d[1] for d in data]
    q1      = [d[2] for d in data]
    median  = [d[3] for d in data]
    q3      = [d[4] for d in data]
    upper   = [d[5] for d in data]
    
    fig = go.Figure(
        go.Box(
            y=names,
            lowerfence=lower,
            q1=q1,
            median=median,
            q3=q3,
            upperfence=upper,
            orientation="h",
            whiskerwidth=1,
            boxpoints=False,
            hoverinfo="y+x",
            notched=True
        )
    )

    for i, m in enumerate(median):
        fig.add_shape(
            type="line",
            x0=m, x1=m,
            y0=i - 0.25, y1=i + 0.25,
            line=dict(color="orange", width=3),
            xref="x", yref="y"
        )
        fig.add_annotation(
            x=m+0.01,
            y=(i - 0.25 + i + 0.25) / 2,
            xref="x",
            yref="y",
            text=f"{m:.2f}",
            showarrow=False,
            font=dict(size=12, color="black"), 
            xanchor="left",
            yanchor="middle"
        )
    
    tick_text = []
    for text in names:
        if "TS_UCF" in text:
            tick_text.append(f'<span style="color: red;">{text}</span>')
        else:
            tick_text.append(f'<span style="color: black;">{text}</span>')

    fig.update_layout(template="plotly_white", width=WIDTH//1.5, height=HEIGHT//1.2, font=FONT_STYLE, font_size=20, margin=dict(l=55, r=15, t=15, b=45, pad=10), showlegend=False, xaxis=dict(
            range=[0, 1],
            tickvals=[0, 0.25, 0.5, 0.75, 1],
            showgrid=True,
            gridcolor="#e5e5e5",
            zeroline=False,
        ),
        yaxis=dict(
            autorange="reversed", 
            showgrid=False,
        ),)
    fig.update_yaxes(tickmode='array', tickvals=list(range(len(names))), ticktext=tick_text)

    if name is not None:
        fig.write_image(f"{name}.pdf", format="pdf")
    
    fig.show()

In [ ]:
def show_box_h(data, name=None):
    names   = [d[0] for d in data]
    lower   = [d[1] for d in data]
    q1      = [d[2] for d in data]
    median  = [d[3] for d in data]
    q3      = [d[4] for d in data]
    upper   = [d[5] for d in data]
    
    fig = go.Figure(
        go.Box(
            x=names, 
            lowerfence=lower,
            q1=q1,
            median=median,
            q3=q3,
            upperfence=upper,
            orientation="v",     
            width=0.75,
            whiskerwidth=1,     
            boxpoints=False,
            hoverinfo="x+y",
            notched=True
        )
    )

    for i, m in enumerate(median):
        fig.add_shape(
            type="line",
            x0=i - 0.35, x1=i + 0.35,
            y0=m, y1=m,    
            line=dict(color="orange", width=3),
            xref="x", yref="y"
        )
        fig.add_annotation(
            x=(i - 0.25 + i + 0.25) / 2,
            y=m + 0.01,
            xref="x",
            yref="y",
            text=f"{m:.2f}",
            showarrow=False,
            font=dict(size=12, color="black"), 
            xanchor="center",
            yanchor="bottom"
        )
    
    tick_text = []
    for text in names:
        if "TS_UCF" in text:
            tick_text.append(f'<span style="color: red;">{text}</span>')
        else:
            tick_text.append(f'<span style="color: black;">{text}</span>')

    fig.update_layout(
        template="plotly_white",
        width=1.2*WIDTH//1.2, height=1.2*HEIGHT//1.5,
        font=FONT_STYLE, font_size=20,
        margin=dict(l=55, r=15, t=15, b=45, pad=10),
        showlegend=False,
        xaxis=dict(
            showgrid=False,
        ),
        yaxis=dict(
            range=[0, 1],
            tickvals=[0, 0.25, 0.5, 0.75, 1],
            showgrid=True,
            gridcolor="#e5e5e5",
            zeroline=False,
        ),
    )
    fig.update_xaxes(tickmode='array', tickvals=list(range(len(names))), ticktext=tick_text, tickangle=-45)

    if name is not None:
        fig.write_image(f"{name}.pdf", format="pdf")
    
    fig.show()

In [ ]:
def plot_bar(pivot: pd.DataFrame, cmap="inferno_r", log_scale=False, name=None):
    row_mins = pivot.min(axis=1).to_numpy()
    row_maxs = pivot.max(axis=1).to_numpy()
    row_ranges = np.where(row_maxs - row_mins == 0, 1, row_maxs - row_mins)
    scaled_pivot = (pivot - row_mins.reshape(-1, 1)) / row_ranges.reshape(-1, 1)

    column_means = pivot.mean(axis=0)

    sorted_columns = column_means.sort_values().index
    pivot_sorted = pivot[sorted_columns]
    scaled_pivot_sorted = scaled_pivot[sorted_columns]


    tick_text = []
    color_bar = []
    for text in sorted_columns:
        if "TS_UCF" in text:
            tick_text.append(f'<span style="color: red;">{text}</span>')
            color_bar.append("red")
        else:
            tick_text.append(f'<span style="color: black;">{text}</span>')
            color_bar.append("blue")

    fig = go.Figure()
    fig.add_trace(
        go.Bar(x=column_means[sorted_columns].index, y=column_means[sorted_columns].values, text=[f'{value:.2f}' for value in column_means[sorted_columns].values], marker_color=color_bar))

    fig.update_layout(template="plotly_white", width=WIDTH, height=HEIGHT//2, font=FONT_STYLE, margin=dict(l=55, r=15, t=15, b=45), showlegend=False)
    
    fig.update_xaxes(side="bottom", tickmode='array', tickvals=list(range(len(sorted_columns))), ticktext=tick_text, showline=True, linecolor='black', tickangle=-45, ticks='outside', showgrid=False)
    fig.update_yaxes(title_text="Average", showline=True, linecolor="black", ticks="inside", mirror=False, gridcolor="#E5E5E5", gridwidth=0.5)

    if name is not None:
        fig.write_image(f"{name}.pdf", format="pdf")
        
    fig.show()

# Plot AUC_ROC Matrices

## Only TS_UCF

In [ ]:
mean_per_dataset = df_concat.groupby(["data_folder", "algo"])["AUC-ROC"].mean().reset_index()
pivot_table = mean_per_dataset.pivot(index="data_folder", columns="algo", values="AUC-ROC")
plot_scaled_matrix(pivot_table)

## Only mTSBench

In [ ]:
plot_scaled_matrix(df_res_mtsbench)

## Both (article figure)

In [ ]:
all_pivot = pd.concat([df_res_mtsbench, pivot_table], axis=1)
plot_scaled_matrix(all_pivot, name=os.path.join(figures_dir, "auc_roc_matrix"))

# Execution time Matrix

In [ ]:
time_mean_per_dataset = df_concat.groupby(["data_folder", "algo"])["runtime"].mean().reset_index()
pivot_table_time = time_mean_per_dataset.pivot(index="data_folder", columns="algo", values="runtime")
plot_scaled_matrix(pivot_table_time, cmap="inferno", log_scale=True)

# VUS-PR Boxplots

## Only MmSBench

In [ ]:
show_box(vus_pr_mtsbench_data)

## Both (article figure)

In [ ]:
vus_pr_ts_ucf_data = list(df_concat.groupby("algo")["VUS-PR"].agg(["min", lambda x: x.quantile(0.25), lambda x: x.median(), lambda x: x.quantile(0.75), "max"]).reset_index().itertuples(index=False,name=None))

In [ ]:
all_vus_pr_data = vus_pr_mtsbench_data.copy()
all_vus_pr_data.extend(vus_pr_ts_ucf_data)
all_vus_pr_data = sorted(all_vus_pr_data, key=lambda x: x[3]) # Sort by median value

In [ ]:
show_box_h(all_vus_pr_data, name=os.path.join(figures_dir, "vus_pr_boxplot"))

# Barplot only on data from NASA

In [ ]:
plot_bar(all_pivot.loc[["MSL", "SMAP"]])

# AC@k table

In [ ]:
from IPython.display import display, Math

In [ ]:
ack_results = pd.read_csv(os.path.join(res_dir, "impact_score_res.csv"))

In [6]:
df_formatted = ack_results.T.astype(object)

# Format the 1st row (index 0) as integers
df_formatted.iloc[0] = df_formatted.iloc[0].apply(lambda x: f"{int(x)}")

# Format the 2nd row (index 1) to 4 decimal places
df_formatted.iloc[1] = df_formatted.iloc[1].apply(lambda x: f"{x:.4f}")

latex_table = (
    df_formatted.to_latex(index=True, header=False, column_format="ccc")
    .replace(r"\begin{tabular}", r"\begin{array}")
    .replace(r"\end{tabular}", r"\end{array}")
    .replace(r"\toprule", r"\hline")
    .replace(r"\midrule", r"\hline")
    .replace(r"\bottomrule", r"\hline")
)
display(Math(latex_table))


<IPython.core.display.Math object>